In [1]:
from pathlib import Path
import typer
from mis_dro.constants import NUM_POSTERIOR_SAMPLES
from datetime import datetime
from mis_dro.dataset import *
import pandas as pd
import math

In [2]:
CONTAMINATION_LEVEL = 0.2  # ratio for contamination dataset
NUM_OBSERVATIONS = 20  # in-sample 'training' observations
NUM_POSTERIOR_SAMPLES = 100  # theta samples from posterior
NUM_TEST_OBSERVATIONS = 50  # out-of-sample 'test' observations
NUM_LIKELIHOOD_SAMPLES = 100  # xi samples from likelihood
NUM_REPLICATIONS = 200  # num times to repeat for loop
NUM_CERTIFY = 200 # num certufying points for discretisation of KDRO problem constraints
MAX_PARAMS_OOM = 1000   # if the number of params of a cvxpy exceeds this number, we might go out-of-memory
IN_SAMPLE_TIME_WINDOW = 52
OUT_OF_SAMPLE_TIME_WINDOW = 12  # number of weeks in out-of-sample period

In [8]:
def upper_triangular_size(dim: int) -> int:
    """Includes the diagonal!"""
    return int(dim * (dim-1) / 2 + dim)

In [51]:
class multivariate_GaussianModel:
    def __init__(self, m, d, known_cov):
        self.m = m
        self.d = d
        self.known_cov = known_cov
    
    def sample(self, theta, key):
        if self.known_cov == True:
            mu = theta
            sigma = DGP_STD_TRUNCATED_NORMAL
            x = (
                jax.random.multivariate_normal(key, mean = mu, cov = (sigma**2)*jnp.eye(self.d), shape=(self.m,self.d))
            )
        else:
            mu = theta[:self.d]
            vec_triu = theta[self.d:]
            Sigma = self.cholesky_param_to_covariance(vec_triu)
            x = (
                jax.random.multivariate_normal(key, mean = mu, cov = Sigma, shape=(self.m,self.d))
            )
        return x
    
    def init_params(self, data):
        if self.known_cov == True:
            return  jnp.mean(data, axis=0).reshape((self.d,))
        else:
            mu_init = jnp.mean(data, axis=0).reshape((self.d,))
            cov_matrix = jnp.cov(data, rowvar=False)
            L = jnp.linalg.cholesky(cov_matrix)
            diag_idx = jnp.diag_indices(L.shape[0])
            diag_entries = L[diag_idx]
            L = L.at[diag_idx].set(jnp.log(diag_entries))
            vec_triu_init = L[jnp.tril_indices(L.shape[0])]
            triu_size = upper_triangular_size(self.d)
            theta_init = jnp.zeros(self.d + triu_size)
            theta_init = theta_init.at[:self.d].set(mu_init)
            theta_init = theta_init.at[self.d:].set(vec_triu_init)
            return theta_init
    
    def parametrise(self, theta):
        # FIXME needs to match what likelihood takes as argument!
        return theta
    
    def reconstruct_covariance_from_triu(self, vec_triu: jnp.array):
        """Reconstruct the covariance matrix from a upper triangular vector in JAX"""
        X = jnp.zeros((self.d, self.d))
        X = X.at[jnp.triu_indices(self.d)].set(vec_triu)
        return X + X.T - jnp.diag(jnp.diag(X))
    
    def cholesky_param_to_covariance(self, L_flat):
        """
        Converts a flattened lower triangular matrix to a covariance matrix.

        L_flat: The flattened lower triangular part of the matrix.
        dim: The dimensionality of the covariance matrix.
        """
        # Reshape the flat array into a lower triangular matrix
        L = jnp.zeros((self.d, self.d))
        tril_indices = jnp.tril_indices(self.d)
        L = L.at[tril_indices].set(L_flat)

        # Diagonal elements of L should be strictly positive to ensure positive definiteness
        L = L.at[jnp.diag_indices(self.d)].set(jnp.exp(jnp.diag(L)))

        # Return the covariance matrix
        return L @ L.T


In [58]:
"""NPL posterior"""

from typing import Optional
import numpy as np
from joblib import Parallel, delayed
from scipy.stats import dirichlet
import scipy.spatial.distance as distance
import itertools
from tqdm import tqdm
import jax
from jax import numpy as jnp
from jax import vmap, value_and_grad, jit, config
from jax.example_libraries import optimizers
from mis_dro.gaussian_kernel import k, k_jax, k_comp


def sample_npl(
    data: np.ndarray,
    inference: str,
    likelihood: str,
    num_posterior_samples: int,
    seed: int,
    lengthscale: float = -1.0,
    dim: int = 1,
    generator: Optional[np.random.Generator] = None,
    p: int = 1,
) -> np.ndarray:
    """Wrapper function for sampling from the NPL posterior with either WLL or MMD loss function

    Args:
        data: Observations sampled from DGP
        inference: Either 'npl_wll' or 'npl_mmd'
        likelihood: Form of likelihood, e.g. 'exponential'
        num_posterior_samples: Number of times to sample from posterior
        generator: numpy random generator
        p: numbers of unknown parameters

    Returns:
        Array of size `num_posterior_samples`
    """
    # NPL posterior sample for theta
    m = data.shape[0]
    if likelihood == "exponential":
        model = ExponentialModel(m)
        p = 1
        dim = 1
    elif likelihood == "normal":
        model = univariate_GaussianModel(m)
        p = 2
        dim = 1
    elif likelihood == "gaussian_known_var":
        model = univariate_GaussianModel_known_variance(m)
        p = 1
    elif likelihood == "multivariate_normal_known_cov":
        d = data.shape[1]
        model = multivariate_GaussianModel(m, d, known_cov=True)
        p = d
    elif likelihood == "multivariate_normal":
        d = data.shape[1]
        model = multivariate_GaussianModel(m, d, known_cov=False)
        p = d + upper_triangular_size(d)
        print(p)
    else:
        raise NotImplementedError(
            f"Posterior '{likelihood}' is not implemented for '{inference}' inference."
        )
    npl_toy = Npl(
        data.reshape((data.shape[0], dim)),
        num_posterior_samples,
        p,
        m,
        model,
        seed,
        l=lengthscale,
        loss_fn=inference,
    )
    npl_toy.draw_samples(random_state=generator)
    theta_sample = npl_toy.sample
    return theta_sample


class Npl:
    """This class contains functions to perform NPL inference (for alpha = 0 in the DP prior) for the Exponential distribution model."""

    def __init__(self, X, B, p, m, model, seed, l=-1, loss_fn="npl_wlb"):
        """
        Args:
            X: Data set
            B: number of bootstrap iterations
            p: number of unknown parameters
            m: number of points sampled from the model at each approximation of the MMD (compatible with value of m within model class)
            l: lengthscale of gaussian kernel; set l = -1 to use median heuristic
            model: model class from models.py
            loss_fn : string set to 'wll' or 'mmd' to specify either the negative log-lkh or mmd-based loss function
        """
        self.B = B
        self.X = X
        self.p = p
        self.loss_fn = loss_fn
        self.n, self.d = self.X.shape
        self.m = m
        self.l = l
        if self.l == -1:  # median heuristic
            self.l = np.sqrt(
                (1 / 2) * np.median(distance.cdist(self.X, self.X, "sqeuclidean"))
            )
        self.kxx = k(
            self.X, self.X, self.l
        )  # pre calculate kernel matrix of data k(x,x)
        self.model = model
        self.seed = seed

    def draw_single_mmd_sample(self, weights, key):
        """Draws a single sample from the nonparametric posterior specified via
        data X and Dirichlet weights"""

        return self.minimise_MMD(self.X, weights, key, eta=0.01)

    def draw_samples(self, n_jobs: int = -1, random_state=None):
        """Draws B samples in parallel from the nonparametric posterior"""

        weights = dirichlet.rvs(np.ones(self.n), size=self.B, random_state=random_state)
        samples = np.zeros((self.B, self.p))

        if self.loss_fn == "npl_wlb":
            # FIXME n_jobs > 1
            temp = Parallel(
                n_jobs=n_jobs,
                backend="multiprocessing",
                max_nbytes=None,
                batch_size="auto",
            )(delayed(self.WLL)(self.X, weights[i, :]) for i in range(self.B))

            for i in range(self.B):
                samples[i, :] = temp[i]
                self.sample = np.array(samples)
        elif self.loss_fn == "npl_mmd":
            key = jax.random.PRNGKey(self.seed)
            key, *subkeys = jax.random.split(key, num=self.B+1) # generate B random keys

            mmd_samples = vmap(self.draw_single_mmd_sample, in_axes=0)(
                weights, jnp.array(subkeys)
            )
            self.sample = np.array(mmd_samples)

    def WLL(self, data, weights):
        """Get weighted negative log likelihood minimizer, for Exponential distribution model"""

        theta = np.zeros(self.d)
        for i in range(self.n):
            theta += weights[i] * data[i, :]
        return 1 / theta

    def MMD_approx(self, kxy, kyy):
        """Approximation of the squared MMD given Gram matrices kxy and kyy"""

        # first sum
        diag_elements = jnp.diag_indices_from(kyy)
        kyy = kyy.at[diag_elements].set(jnp.repeat(0, self.m))
        sum1 = jnp.sum(kyy)

        # second sum
        sum2 = jnp.sum(kxy)

        # third sum
        diag_elements = jnp.diag_indices_from(self.kxx)
        kxx = self.kxx.at[diag_elements].set(jnp.repeat(0, self.n))
        sum3 = jnp.sum(kxx)

        return (
            (1 / (self.m * (self.m - 1))) * sum1
            - (2 / (self.n * self.m)) * sum2
            + (1 / (self.n * (self.n - 1))) * sum3
        )

    def minimise_MMD(self, data, weights, key, Nstep=1000, eta=0.1, batch_size=10):
        """Function to minimise the MMD using adam optimisation in JAX"""

        key, key1, key2 = jax.random.split(key, num=2 + 1)
        params = self.model.init_params(data)
        config.update("jax_enable_x64", True)
        num_batches = self.n // batch_size

        # objective function to feed the optimizer
        def obj_fun(theta, x, n, key):
            y = self.model.sample(
                theta, key
            )  # Returnes self.m random samples from the model with parameter theta

            if self.d > 1:
                # Compute kernel Gram matrices
                kyy = k_comp(y, y) #, self.l
                kxy = k_comp(y, x) #, self.l
            else:
                kyy = k_jax(y, y, self.l)
                kxy = k_jax(y, x, self.l)

            # first sum
            diag_elements = jnp.diag_indices_from(kyy)
            kyy = kyy.at[diag_elements].set(jnp.repeat(0, self.m))
            sum1 = jnp.sum(kyy)

            # second sum
            sum2 = jnp.sum(kxy)

            # Return first two terms of squared MMD; note that the third term does not depend on theta!
            return (1 / (self.m * (self.m - 1))) * sum1 - (2 / (n * self.m)) * sum2

        opt_init, opt_update, get_params = optimizers.adam(step_size=eta)
        opt_state = opt_init(params)
        itercount = itertools.count()

        # Define gradient function
        grad_fn = vmap(
            jit(value_and_grad(obj_fun, argnums=0)), in_axes=(None, 0, None, None)
        )

        # Function to evaluate gradient and loss value at each step
        def step(step, opt_state, batches, key):
            key, subkey = jax.random.split(key)
            values, grads = grad_fn(get_params(opt_state), batches, batch_size, subkey)
            opt_state = opt_update(step, np.mean(grads, axis=0), opt_state)
            value = np.mean(values, axis=0)
            return value, opt_state

        smallest_loss = 1000000
        best_theta = get_params(opt_state)
        key1, *rng_inputs1 = jax.random.split(key1, num=Nstep + 1)
        key2, *rng_inputs2 = jax.random.split(key2, num=Nstep + 1)
        for i in range(Nstep):
            batches = []
            _, *rng_inputs = jax.random.split(rng_inputs2[i], num=num_batches + 1)
            for j in range(num_batches):
              inds = jax.random.choice(rng_inputs[j], a=self.n, shape=(batch_size,), p=weights) #default is with replacement
              batch_x = jnp.take(a=data, indices=inds, axis=0)
              batches.append(batch_x)

            batches = jnp.array(batches)
            # Update loss and gradient
            value, opt_state = step(next(itercount), opt_state, batches, rng_inputs1[i])
            print(get_params(opt_state))
            # Update smallest loss and best theta value if loss has decreased
            pred = value < smallest_loss  # Prediction that loss (value) has decreased

            def true_func(args):
                value, smallest_loss, best_theta, opt_state = (
                    args[0],
                    args[1],
                    args[2],
                    args[3],
                )
                smallest_loss = value
                best_theta = get_params(opt_state)
                return smallest_loss, best_theta

            def false_func(args):
                value, smallest_loss, best_theta, opt_state = (
                    args[0],
                    args[1],
                    args[2],
                    args[3],
                )
                smallest_loss = jnp.array(smallest_loss, dtype="float64")
                return smallest_loss, best_theta

            # Updates smallest loss and best theta if prediction (pred) is correct
            smallest_loss, best_theta = jax.lax.cond(
                pred,
                true_func,
                false_func,
                [value, smallest_loss, best_theta, opt_state],
            )
            
            
       
        best_theta = self.model.parametrise(best_theta)
        return best_theta 

In [59]:
def portfolio_dataset(dgp: str, time_window_id: int, mmc2_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    """Gets the training and test datasets for the porfolio problem.

    Args:
        dgp: Options include 'DowJones', 'FF49Industries', 'FTSE100', 'NASDAQ100', 'NASDAQComp', 'SP500'
        time_window_id: The ID of the time window.
        mmc2_dir: Path to the directory downloaded from 'data-in-brief' webpage below

    Returns:
        training_data: numpy array of shape (NUM_TRADING_DAYS_IN_YEAR, NUM_STOCKS)
        test_data: numpy array of shape (NUM_TRADING_DAYS_IN_QUARTER, NUM_STOCKS)

    Notes:
        Download data from https://www.data-in-brief.com/article/S2352-3409(16)30399-7/fulltext
    """
    returns_df = pd.read_excel(mmc2_dir / f"{dgp}.xlsx", sheet_name="Assets_Returns", header=None)
    num_time_windows = get_num_time_windows(len(returns_df))
    assert time_window_id < num_time_windows
    start_training_week = time_window_id * OUT_OF_SAMPLE_TIME_WINDOW # inclusive
    end_training_week = start_training_week + IN_SAMPLE_TIME_WINDOW # not inclusive
    start_test_week = end_training_week # inclusive
    end_test_week = start_test_week + OUT_OF_SAMPLE_TIME_WINDOW # not inclusive
    training_data = returns_df.iloc[start_training_week: end_training_week].values
    test_data = returns_df.iloc[start_test_week: end_test_week].values
    return training_data, test_data

def get_num_time_windows(num_weeks: int) -> int:
    return math.floor((num_weeks - IN_SAMPLE_TIME_WINDOW) / OUT_OF_SAMPLE_TIME_WINDOW)

In [60]:
def portfolio_sample_npl(
    replication: int,
    dataset_dir: Path,
    dgp: str = "DowJones",
    inference: str = "npl_mmd",
    lengthscale: float = -1.0,
    likelihood: str = "multivariate_normal",
    num_posterior_samples: int = NUM_POSTERIOR_SAMPLES,
):
    """Run a single replication where the seed is given by the replication number"""
    # 1. load portfolio dataset
    generator = np.random.default_rng(seed=replication)
    data, data_eval = portfolio_dataset(dgp, replication, dataset_dir)
    print(data.shape)

    # 2. sample from the posterior
    print()
    print(datetime.now(), "- Starting portfolio sample NPL for replication", replication)    
    theta_sample = sample_npl(
        data,
        inference,
        likelihood,
        num_posterior_samples,
        seed=replication,
        lengthscale=lengthscale,
        generator=generator,
        dim=data.shape[1],
    )
    return theta_sample
    # df = pd.DataFrame(theta_sample)
    # df.to_csv(experiment_dir / f"portfolio_theta_sample_{dgp}_{replication}.csv", index=False, header=False)

In [61]:
theta_sample = portfolio_sample_npl(
    0,
    Path("/dcs/pg23/u1604520/misdro/"),
    num_posterior_samples=10    # FIXME is this correct?
)

(52, 28)

2024-10-06 12:58:31.031581 - Starting portfolio sample NPL for replication 0
434
406
Traced<ShapedArray(float64[434])>with<BatchTrace(level=1/0)> with
  val = Array([[ 2.14460108e-03,  6.50309775e-03,  1.23875181e-02, ...,
        -2.89318932e-03, -7.75874593e-03, -3.92410261e+00],
       [ 2.21445995e-02,  6.50309772e-03, -7.61247258e-03, ...,
        -2.89319036e-03, -7.75875266e-03, -3.92410264e+00],
       [ 2.21445999e-02,  6.50311617e-03, -7.61247532e-03, ...,
        -2.89318987e-03,  1.22411004e-02, -3.92410262e+00],
       ...,
       [ 2.14460223e-03,  6.50309889e-03, -7.61247499e-03, ...,
        -2.89318664e-03, -7.75874677e-03, -3.92410258e+00],
       [ 2.21445999e-02,  6.50309959e-03, -7.61247329e-03, ...,
        -2.89318950e-03, -7.75872882e-03, -3.92410263e+00],
       [ 2.21445995e-02,  2.65030963e-02, -7.61247438e-03, ...,
        -2.89314849e-03,  1.22411921e-02, -3.92410246e+00]],      dtype=float64)
  batch_dim = 0


KeyboardInterrupt: 

In [45]:
print(theta_sample.shape)
theta_mean = np.array(theta_sample.mean(axis=0))

(10, 434)


In [37]:
def cholesky_param_to_covariance(dim, L_flat):
    # Reshape the flat array into a lower triangular matrix
    L = np.zeros((dim, dim))
    tril_indices = np.tril_indices(dim)
    L[tril_indices] = L_flat

    # Diagonal elements of L should be strictly positive to ensure positive definiteness
    L[np.diag_indices(dim)] = np.exp(jnp.diag(L))

    # Return the covariance matrix
    return L @ L.T

def reparametrise_NPL_output(dim, theta):
    mu = theta[:dim]
    L_flat = theta[dim:]
    cov = cholesky_param_to_covariance(dim, L_flat)
    return mu, cov

In [40]:
theta_mean.shape

(434,)

In [62]:
dim = 28
print(int(dim * (dim-1) / 2 + dim))

406


In [63]:
reparametrise_NPL_output(28, theta_mean)

(array([ 0.01738085,  0.01760344, -0.00052836,  0.01244125,  0.00489844,
         0.01239415, -0.00124654,  0.00107403,  0.01313097,  0.01046449,
         0.01152654,  0.00412851,  0.02580533,  0.00530422,  0.00447397,
         0.00488305,  0.01200539,  0.01888082,  0.03338574,  0.00319623,
         0.00681847,  0.02569318,  0.00740784,  0.00186839,  0.00222441,
         0.00084181,  0.00366765,  0.00337864]),
 array([[ 3.83693735e-03,  1.34512211e-04, -6.38663988e-06,
          7.17968930e-05,  9.94674226e-05,  1.03203782e-04,
          4.05875205e-04, -8.73659020e-06,  2.13582098e-04,
          3.74281384e-05,  3.26459657e-05,  5.17538115e-05,
          1.76662742e-04,  2.30590788e-04,  5.55206478e-04,
          1.85872378e-04,  1.99471801e-04,  1.72050044e-04,
          1.86627521e-04,  1.84854700e-04,  1.29391867e-04,
          4.61828543e-04,  2.18056888e-04,  1.22926004e-04,
          3.03711509e-04,  6.84636724e-05,  2.15288894e-04,
          1.50998156e-04],
        [ 1.3451221

In [14]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
